# Radio Bulletin Generator — Three Languages

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

Generate natural, radio-ready weather bulletins in **three languages**:
1. **English** (Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

- **Target length**: ~750 words (~5 minutes at 150 wpm radio reading pace)
- **Style**: Philippine radio broadcast — authoritative, clear, flowing prose
- **Model**: Gemma 4 E4B via Ollama (text-only — no vision needed here)
- **Input**: Full markdown bulletin text (from notebook 04)
- **Output**: Plain text scripts saved to `data/radio_bulletins/`

### Two sample bulletins
1. `PAGASA_20-19W_Pepito_SWB#01` — Severe Weather Bulletin, Tropical Depression Pepito
2. `PAGASA_22-TC02_Basyang_TCA#01` — Tropical Cyclone Alert, Basyang

**Total output**: 6 scripts (2 bulletins × 3 languages)

In [1]:
import json
import requests
import time
from pathlib import Path

OLLAMA_API = "http://localhost:11434"
MODEL_NAME = "gemma4:e4b"
TIMEOUT = 10 * 60  # 10 minutes

data_dir = Path("../data")
markdown_dir = data_dir / "gemma4_results"
output_dir = data_dir / "radio_bulletins"
output_dir.mkdir(exist_ok=True)

# The two bulletins we are working with
SAMPLES = [
    "PAGASA_20-19W_Pepito_SWB#01",
    "PAGASA_22-TC02_Basyang_TCA#01",
]

# Verify Ollama is running
try:
    r = requests.get(f"{OLLAMA_API}/api/tags", timeout=5)
    names = [m['name'] for m in r.json()['models']]
    status = "\u2713" if any(MODEL_NAME in n for n in names) else "\u26a0\ufe0f  model not found"
    print(f"\u2713 Ollama running \u2014 {MODEL_NAME}: {status}")
except Exception as e:
    print(f"\u2717 Ollama not reachable: {e}")

print(f"\u2713 Output dir: {output_dir.absolute()}")


✓ Ollama running — gemma4:e4b: ✓
✓ Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins


## 1. Load Bulletin Markdown

Load the raw markdown text extracted by notebook 04. This is the primary input — no structured JSON used.

In [2]:
def load_bulletin(stem: str) -> dict:
    """Load the raw markdown for a bulletin stem (primary input for generation)."""
    markdown_path = markdown_dir / f"{stem}_markdown.md"

    if not markdown_path.exists():
        raise FileNotFoundError(f"Markdown file not found: {markdown_path}")

    markdown = markdown_path.read_text(encoding="utf-8")

    return {"stem": stem, "markdown": markdown}


bulletins = [load_bulletin(s) for s in SAMPLES]

for b in bulletins:
    print(f"\n{b['stem']}")
    print(f"  Markdown: {len(b['markdown'])} chars")
    print(f"  Preview:  {b['markdown'][:120].strip()!r}")



PAGASA_20-19W_Pepito_SWB#01
  Markdown: 4825 chars
  Preview:  '## REPUBLIC OF THE PHILIPPINES\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PHILIPPINE ASTROPHYSICAL, GEOPHYSICAL AND ASTR'

PAGASA_22-TC02_Basyang_TCA#01
  Markdown: 3224 chars
  Preview:  '# PHILIPPINE ARCHIPELAGO\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PAGASA**\nPhilippine Atmospheric, Geophysical and Ast'


## 2. Radio Bulletin Prompts — Three Languages

Generate radio bulletins in:
1. **English** (standard Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

All versions maintain the same style:
- Flowing prose — no bullet points, no tables, no markdown

- ~750 words (~5 minutes reading time)- Clear structure: opening → current situation → forecast → affected areas → public safety → closing

- Repeat storm name and signal areas (radio listeners may tune in mid-broadcast)- Plain spoken numbers

In [3]:
PROMPTS = {
    "en": {
        "system": """You are a Philippine radio weather broadcaster writing a spoken bulletin script for on-air reading in English.

STYLE RULES:
- Write in flowing, natural prose suitable for reading aloud.
- Use a calm, authoritative tone appropriate for public emergency broadcasts.
- Spell out numbers when they will be read aloud (e.g. "one hundred thirty kilometres per hour", "Signal Number Two").
- Repeat the storm name and the most critical warning signal at least twice — radio listeners may tune in partway through.
- Use natural spoken transitions: "At this time...", "Moving on to the forecast...", "Residents are urged to..."
- Reference the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA, by name at the start.
- Close with the time of the next scheduled bulletin update.

FORMATTING: Output in Markdown so the script is easy to read and review.
- Use a top-level heading for the bulletin title (storm name + bulletin type)
- Use second-level headings for each section: Current Situation, Forecast Track, Affected Areas, Public Safety Advisory, Closing
- Bold the storm name and signal numbers on first mention in each section
- Keep the prose itself natural and speakable — the markdown is for readability only

LENGTH: Target approximately 750 words — this should read aloud in about five minutes at a steady broadcast pace.""",

        "user_template": (
            "Write a five-minute radio broadcast bulletin script in English "
            "based on the following PAGASA weather bulletin text.\n\n"
            "{markdown}\n\n"
            "Write the radio script now. Use Markdown formatting, approximately 750 words."
        ),
    },

    "tl": {
        "system": """Ikaw ay isang Filipino radio weather broadcaster na sumusulat ng spoken bulletin script para basahin sa ere sa Tagalog.

MGA PATAKARAN SA ESTILO:
- Sumulat sa natural na daloy ng prosa na angkop para basahin nang malakas.
- Gumamit ng kalmado at may awtoridad na tono na angkop para sa public emergency broadcasts.
- I-spell out ang mga numero kapag babasahin (halimbawa: "isandaan at tatlumpung kilometro bawat oras", "Signal Number Two").
- Ulitin ang pangalan ng bagyo at ang pinaka-kritikal na babala nang kahit dalawang beses.
- Gumamit ng natural na transisyon: "Sa ngayon...", "Patungo sa forecast...", "Hinihikayat ang mga residente na..."
- Banggitin ang PAGASA sa simula ng bulletin.
- Magtapos sa oras ng susunod na scheduled bulletin update.

FORMATTING: Mag-output sa Markdown para madaling basahin at suriin.
- Gumamit ng top-level heading para sa pamagat ng bulletin (pangalan ng bagyo + uri ng bulletin)
- Gumamit ng second-level headings para sa bawat seksyon: Kasalukuyang Sitwasyon, Forecast Track, Mga Apektadong Lugar, Payo sa Kaligtasan, Pangwakas
- I-bold ang pangalan ng bagyo at signal numbers sa unang pagbanggit sa bawat seksyon
- Panatilihing natural ang prosa para mabasa nang maayos

HABA: Target na humigit-kumulang 750 salita — dapat itong mabasa sa loob ng limang minuto.""",

        "user_template": (
            "Sumulat ng limang minutong radio broadcast bulletin script sa Tagalog "
            "batay sa sumusunod na PAGASA weather bulletin text.\n\n"
            "{markdown}\n\n"
            "Sumulat ng radio script ngayon. Gumamit ng Markdown formatting, humigit-kumulang 750 salita."
        ),
    },

    "ceb": {
        "system": """Ikaw usa ka Filipino radio weather broadcaster nga direkta nakigsulti sa mga mamumuno sa Cebuano. Isulat ang bulletin sama sa imong gisulti sa radyo — natural, conversational, ug dali masabtan.

MGA LAGDA SA ESTILO (CONVERSATIONAL RADIO CEBUANO):
- Pagsulat sama sa regular nga radyo broadcaster nga nakigsulti sa mga tagapaminaw — dili formal, natural lang nga Bisaya.
- Gamita ang mga common nga radio phrases: "Ato pang hisgutan...", "Karon, mga higala...", "Unya ha...", "Importante nga..."
- I-spell out ang mga numero sama sa pagsulti (pananglitan: "usa ka gatos ug katluan ka kilometro matag oras", "Signal Number Two").
- Sublia ang ngalan sa bagyo ug signal areas og duha o tulo ka beses — basin lang dili makahibalo ang uban.
- Gamita ang conversational nga transitions: "Karon ha...", "Unya kini...", "Importante kaayo nga...", "Mga kauban..."
- Mention PAGASA sa sinugdan pero dili kaayo formal — natural lang.
- Tapuson sama sa regular nga radyo: "Tan-awa nato pag-usab sa..." o "Magkita tag usab sa..."

FORMATTING: Mag-output sa Markdown aron dali mabasahan.
- Paggamit og top-level heading alang sa titulo sa bulletin (ngalan sa bagyo + matang sa bulletin)
- Paggamit og second-level headings alang sa matag seksyon pero simple lang: Unsay Nahitabo Karon, Asa Padulong ang Bagyo, Kinsa ang Apektado, Unsa ang Kinahanglan Buhaton, Pangwakas
- I-bold ang ngalan sa bagyo ug signal numbers
- Pero ang script mismo kinahanglan conversational — parang nag-istambay lang mo sa radyo

GITAS-ON: Mga 750 ka pulong — lima ka minuto nga radyo talk, dili basahon nga formal.""",

        "user_template": (
            "Isulat ang lima ka minutong conversational radio bulletin sa Cebuano "
            "base sa PAGASA weather bulletin text nga naay sulod dinhi.\n\n"
            "{markdown}\n\n"
            "Pagsulat karon — conversational Bisaya, parang nag-istorya lang sa radyo. Markdown format, mga 750 ka pulong."
        ),
    },
}


def build_user_prompt(bulletin: dict, language: str) -> str:
    """Build the user prompt using the full markdown bulletin text as input."""
    template = PROMPTS[language]["user_template"]
    return template.format(markdown=bulletin["markdown"])


print("\u2713 Prompts defined for 3 languages")
for lang, prompts in PROMPTS.items():
    print(f"  {lang}: system={len(prompts['system'])} chars, template={len(prompts['user_template'])} chars")


✓ Prompts defined for 3 languages
  en: system=1362 chars, template=206 chars
  tl: system=1304 chars, template=227 chars
  ceb: system=1579 chars, template=250 chars


## 3. Generate Radio Bulletins — All Three Languages

Call Gemma 4 for each bulletin in English, Tagalog, and Cebuano (6 total scripts).

In [4]:
def generate_radio_bulletin(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a radio broadcast script for one bulletin in the specified language."""
    stem = bulletin["stem"]
    lang_names = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}
    print(f"\nGenerating: {stem} ({lang_names[language]})")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": PROMPTS[language]["system"]},
            {"role": "user", "content": build_user_prompt(bulletin, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    elapsed = time.time() - t_start

    script = response.json().get("message", {}).get("content", "").strip()

    # Word count (strip markdown syntax for accurate spoken-word estimate)
    import re
    plain = re.sub(r"[#*_`>\-]", "", script)
    word_count = len(plain.split())
    reading_minutes = word_count / 150  # ~150 wpm for radio

    # Save as markdown file
    out_path = output_dir / f"{stem}_radio_{language}.md"
    out_path.write_text(script, encoding="utf-8")

    print(f"  \u2713 Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}  |  Est. reading time: {reading_minutes:.1f} min")
    print(f"  Saved \u2192 {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "script": script,
        "word_count": word_count,
        "reading_minutes": reading_minutes,
        "elapsed": elapsed,
    }


results = []
for bulletin in bulletins:
    for lang in ["en", "tl", "ceb"]:
        result = generate_radio_bulletin(bulletin, lang)
        results.append(result)

print(f"\n\u2713 Done \u2014 {len(results)} scripts generated ({len(bulletins)} bulletins \u00d7 3 languages)")



Generating: PAGASA_20-19W_Pepito_SWB#01 (English)
  ✓ Generated in 50.3s
  Words: 608  |  Est. reading time: 4.1 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_en.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Tagalog)
  ✓ Generated in 60.0s
  Words: 697  |  Est. reading time: 4.6 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_tl.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Cebuano)
  ✓ Generated in 64.4s
  Words: 705  |  Est. reading time: 4.7 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_ceb.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (English)
  ✓ Generated in 44.1s
  Words: 601  |  Est. reading time: 4.0 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_en.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Tagalog)
  ✓ Generated in 62.8s
  Words: 707  |  Est. reading time: 4.7 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_tl.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Cebuano)
  ✓ Generated in 56.8s
  Words: 694  |  Est. reading time: 4.6 min
  Saved → PAGASA_22-TC02_Basyan

## 6. TTS Plain Text Generation — All Three Languages

Generate TTS-optimized plain text versions of all radio scripts via a second
Gemma4 prompt. These files feed directly into notebook 08 — no markdown stripping needed.

**Output:** `data/radio_bulletins/{stem}_tts_{lang}.txt` (CEB, TL, EN)

In [5]:
TTS_PROMPTS = {
    "ceb": {
        "system": """Ikaw usa ka espesyalista sa Cebuano nga nagsulat og plain text nga angay para sa text-to-speech synthesis.

Ang imong trabaho:
- Basaha ang markdown radio script nga gihatag
- Isulat kini pag-usab isip natural nga flowing prose SA CEBUANO LAMANG — walay markdown
- WALA markdown: wala headings (#), wala bullet points (-), wala asterisks (*), wala bold/italic
- Para sa mga English proper nouns o teknikal nga termino, i-spell sila phonetically sa Cebuano:
  - PAGASA → pa-ga-sa
  - Northern Luzon → nor-dern lu-son
  - Signal Number One / Two / Three → sig-nal nam-ber wan / tu / tri
  - tropical depression → tro-pi-kal di-pre-syon
  - tropical storm → tro-pi-kal storm
  - kilometers per hour → ki-lo-me-tros sa usa ka oras
  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west
- Pahimusa ang paragraph structure: blank lines tali sa mga paragraph
- AYAW pagdugang og bisan unsa nga texto nga wala sa orihinal nga script
- Output: plain text lamang, walay bisan unsang markup o formatting characters""",

        "user_template": (
            "Basaha kining markdown radio script ug isulat kini pag-usab isip TTS-ready plain Cebuano text.\n\n"
            "{markdown}\n\n"
            "Isulat ang plain Cebuano text karon. Cebuano nga pulong lamang, phonetically spelled kung "
            "kinahanglan, paragraph breaks (blank lines) para sa natural nga pausing. Walay markdown."
        ),
    },
    "tl": {
        "system": """Ikaw ay isang espesyalista sa Tagalog na sumusulat ng plain text na angkop para sa text-to-speech synthesis.

Ang iyong trabaho:
- Basahin ang markdown radio script na ibinigay
- Isulat muli ito bilang natural na flowing prose SA TAGALOG LAMANG — walang markdown
- WALANG markdown: walang headings (#), walang bullet points (-), walang asterisks (*), walang bold/italic
- Para sa mga English proper nouns o teknikal na termino, i-spell ang mga ito nang phonetically sa Tagalog:
  - PAGASA → pa-ga-sa
  - Northern Luzon → nor-dern lu-son
  - Signal Number One / Two / Three → sig-nal nam-ber wan / tu / tri
  - tropical depression → tro-pi-kal di-pre-syon
  - tropical storm → tro-pi-kal storm
  - kilometers per hour → ki-lo-me-tro ba-wat o-ras
  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west
- Panatilihin ang paragraph structure: blank lines sa pagitan ng mga paragraph
- HUWAG magdagdag ng anumang texto na wala sa orihinal na script
- Output: plain text lamang, walang anumang markup o formatting characters""",

        "user_template": (
            "Basahin ang markdown radio script na ito at isulat muli ito bilang TTS-ready plain Tagalog text.\n\n"
            "{markdown}\n\n"
            "Isulat ang plain Tagalog text ngayon. Tagalog na salita lamang, phonetically spelled kung "
            "kinakailangan, paragraph breaks (blank lines) para sa natural na pausing. Walang markdown."
        ),
    },
    "en": {
        "system": """You are a specialist in writing plain text suitable for text-to-speech synthesis.

Your task:
- Read the provided markdown radio script
- Rewrite it as natural flowing prose IN ENGLISH ONLY — no markdown
- NO markdown: no headings (#), no bullet points (-), no asterisks (*), no bold/italic
- Keep technical terms clear and pronounceable
- Maintain paragraph structure: blank lines between paragraphs
- DO NOT add any text that wasn't in the original script
- Output: plain text only, no markup or formatting characters""",

        "user_template": (
            "Read this markdown radio script and rewrite it as TTS-ready plain English text.\n\n"
            "{markdown}\n\n"
            "Write the plain English text now. Clear pronunciation-friendly English, "
            "paragraph breaks (blank lines) for natural pausing. No markdown."
        ),
    },
}


def build_tts_prompt(markdown_script: str, language: str) -> str:
    """Build the user prompt for TTS text generation."""
    return TTS_PROMPTS[language]["user_template"].format(markdown=markdown_script)


print("✓ TTS_PROMPTS defined for CEB, TL, EN")
print(f"  ceb: system={len(TTS_PROMPTS['ceb']['system'])} chars")
print(f"  tl:  system={len(TTS_PROMPTS['tl']['system'])} chars")
print(f"  en:  system={len(TTS_PROMPTS['en']['system'])} chars")

✓ TTS_PROMPTS defined for CEB, TL, EN
  ceb: system=1040 chars
  tl:  system=1055 chars
  en:  system=519 chars


In [6]:
def generate_tts_text(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a TTS-ready plain text script for one bulletin."""
    stem = bulletin["stem"]
    lang_names = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}
    print(f"\nGenerating TTS text: {stem} ({lang_names[language]})")

    markdown_script = (output_dir / f"{stem}_radio_{language}.md").read_text(encoding="utf-8")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": TTS_PROMPTS[language]["system"]},
            {"role": "user", "content": build_tts_prompt(markdown_script, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    elapsed = time.time() - t_start

    tts_text = response.json().get("message", {}).get("content", "").strip()

    out_path = output_dir / f"{stem}_tts_{language}.txt"
    out_path.write_text(tts_text, encoding="utf-8")

    word_count = len(tts_text.split())
    print(f"  ✓ Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}")
    print(f"  Saved → {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "tts_text": tts_text,
        "word_count": word_count,
        "elapsed": elapsed,
    }


# Generate TTS text for both bulletins — all three languages
tts_results = []
for bulletin in bulletins:
    for lang in ["ceb", "tl", "en"]:
        result = generate_tts_text(bulletin, lang)
        tts_results.append(result)

print(f"\n✓ Done — {len(tts_results)} TTS text files generated")

# Preview first bulletin (CEB)
print("\nPREVIEW — CEB TTS text (first 500 chars)")
print("=" * 60)
ceb_result = [r for r in tts_results if r["language"] == "ceb"][0]
print(ceb_result["tts_text"][:500])
print("...")


Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (Cebuano)
  ✓ Generated in 47.2s
  Words: 630
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_ceb.txt

Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (Tagalog)
  ✓ Generated in 59.9s
  Words: 676
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_tl.txt

Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (English)
  ✓ Generated in 20.5s
  Words: 593
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_en.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (Cebuano)
  ✓ Generated in 47.6s
  Words: 641
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_ceb.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (Tagalog)
  ✓ Generated in 48.6s
  Words: 666
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_tl.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (English)
  ✓ Generated in 17.1s
  Words: 572
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_en.txt

✓ Done — 6 TTS text files generated

PREVIEW — CEB TTS text (first 500 chars)
Kumusta na mo diha, mga kauban! Malip

## 4. Review Output

Display each generated script for manual inspection (grouped by bulletin, showing all languages).

In [7]:
from IPython.display import display, Markdown
from collections import defaultdict

grouped = defaultdict(list)
for r in results:
    grouped[r["stem"]].append(r)

lang_labels = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}

for stem, versions in grouped.items():
    display(Markdown(f"---\n# {stem}\n---"))
    for version in versions:
        lang = version["language"]
        meta = (
            f"**Language:** {lang_labels[lang]}  \u2502  "
            f"**Words:** {version['word_count']}  \u2502  "
            f"**Est. reading time:** {version['reading_minutes']:.1f} min"
        )
        display(Markdown(f"## {lang_labels[lang]} Version\n\n{meta}\n\n---\n\n{version['script']}"))


---
# PAGASA_20-19W_Pepito_SWB#01
---

## English Version

**Language:** English  │  **Words:** 608  │  **Est. reading time:** 4.1 min

---

# Tropical Depression PEPITO Advisory Bulletin

## Current Situation
Good morning, this is a severe weather bulletin brought to you by the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA. We are issuing this advisory for **Tropical Depression PEPITO**.

As of five o’clock in the morning this morning, October nineteenth, **Tropical Depression PEPITO** has developed from a low-pressure area situated to the east of Catanduanes. At this time, the storm center is currently estimated to be passing east of Virday, Catanduanes.

Listeners, please be advised that while **Tropical Depression PEPITO** is active, PAGASA has not yet raised any Tropical Cyclone Wind Signal. However, the atmosphere remains highly volatile, and we urge all residents to remain vigilant. PAGASA meteorologists continue to monitor the system closely, as the area has the potential to intensify further into a tropical storm or even a typhoon.

## Forecast Track
Moving on to the forecast track, **Tropical Depression PEPITO** is currently projected to move in a generally west-northwestward to north-northwestward direction today. Over the next day, the storm is forecast to shift its trajectory more westward, heading towards Northern Luzon.

The most critical projection to understand is that **Tropical Depression PEPITO** is forecast to make landfall along the eastern coast of Northern Luzon during the afternoon of October twenty-first.

Looking ahead, the expected movement path suggests that the storm will traverse the waters east of Catanduanes, passing near the eastern coasts of Samar and Leyte, before continuing its westward march toward the main body of Northern Luzon.

## Affected Areas
Residents are urged to pay close attention to the areas that PAGASA has identified as most vulnerable.

The regions that are expected to be most severely affected by the potential impacts of **Tropical Depression PEPITO** include Catanduanes, Northern Samar, Samar, and significant parts of Quezon Province, extending up to Northern Luzon.

For those in the Bicol Region, we anticipate rainfall accompanied by strong winds. Similar conditions are expected in Catanduanes. Across the wider areas of Northern Samar, Samar, Southern Leyte, Masbate, Albay, Camarines Norte, Camarines Sur, and Quezon, residents should brace for adverse weather. Furthermore, coastal areas spanning from Northern Luzon, through Isabela, Cagayan, Ilocos Norte, Aurora, and Zambales, are also expected to experience strong winds and rainfall.

## Public Safety Advisory
Regarding hazards to expect in coastal waters, we must advise extreme caution. Mariners should anticipate moderate to strong seas, potentially reaching between two point five and four point five meters, particularly on the northeast and former south sides. Severe rainfall is anticipated for all those utilizing small sea craft or commercial vessels, and we advise that all nearby waterways remain highly hazardous.

To summarize the hazards, residents in the Bicol Region and Catanduanes should be prepared for rain and strong winds. Again, we stress that Northern Samar, Samar, Southern Leyte, Masbate, Albay, Camarines Norte, Camarines Sur, Quezon, and the general area of Northern Luzon are expected to experience the most severe conditions.

Residents are reminded that although PAGASA has not issued a Tropical Cyclone Wind Signal at this time, the atmosphere is rapidly changing. All local government units, and the disaster risk reduction and management councils, must prepare and take appropriate precautionary measures immediately. We repeat: **Tropical Depression PEPITO** is the storm system of concern, and while no signal is currently raised, residents must remain highly alert and ready to respond to escalating warnings.

## Closing
We will continue to monitor **Tropical Depression PEPITO** and issue updates as necessary. For your safety, please remain tuned to your local radio station for the next Severe Weather Bulletin, which is scheduled to be issued at eleven o’clock in the morning today. Thank you for listening, and please stay safe.

## Tagalog Version

**Language:** Tagalog  │  **Words:** 697  │  **Est. reading time:** 4.6 min

---

# PAGASA Weather Bulletin: Tropical Depression "Pepito" Advisory

## Kasalukuyang Sitwasyon
Magandang umaga po sa ating mga tagapakinig. Ito po ay isang espesyal na pag-uulat mula sa Philippine Atmospheric, Geophysical and Astronomical Services Administration, o PAGASA.

Sa oras na ito, limang kaayauman ng umaga, araw ng ika-labing-siyam ng Oktubre, taong dalawampu at dalawa, nagpapatuloy po ang PAGASA sa pagsubaybay sa isang mababang presyon na lugar na matatagpuan sa silangan ng Catanduanes.

Matapos ang masusing pag-aaral, idineklara na ang lugar na ito ay naging isang **Tropical Depression "Pepito."**

Hinihiling po namin sa lahat na manatiling alerto. Ang **Tropical Depression "Pepito"** ay inaasahang magpapatuloy sa paggalaw, na may posibilidad pa ring maglakas pa, na posibleng lumakas pa upang maging Tropical Storm o kahit Typhoon.

Ayon sa mga forecast, ang inaasahang paglapit ng **Tropical Depression "Pepito"** ay inaasahang makakarating sa baybayin ng hilagang Luzon sa hapon ng ika-dalawampu't isa ng Oktubre.

## Forecast Track at Mga Apektadong Lugar
**Patungo sa forecast** para sa mga sumusunod na oras:

Sa unang pag-uulat ngayong araw, ang sentro ng **Tropical Depression "Pepito"** ay tinatayang nasa walong daan at dalawampung kilometro silangan ng Virday, Catanduanes.

Samantala, para sa susunod na dalawampu't-kwatro oras, o bukas, inaasahang mapapahamak ito sa anim daan kilometro silangan ng Cagayan, Aurora. Pagdating ng tatlumpung-anim na oras, o Miyerkules, inaasahan naman ang paglapit nito sa isang daan at animnapung kilometro silangan ng Pangasinan, Isabela.

Ang inaasahang direksyon ng bagyo ay patungo sa kanluran, at sa loob ng apatnapu't walong oras, o Biyernes ng umaga, inaasahang ito ay nasa isang daan at limampung kilometro kanluran ng Pangasinan City.

Ang paglalakbay ng bagyo ay inaasahang magdadala ng mga babala sa mga sumusunod na rehiyon: Kabilang dito ang Bicol Region, Catanduanes, at ang pinaka-malalakas na apektado ay inaasahang mga lugar ng Northern Samar, Samar, Southern Leyte, Masbate, Albay, Camarines Norte, Camarines Sur, at Quezon.

Ang mga lugar sa baybayin ng hilagang Luzon, kabilang ang Isabela, Cagayan, Ilocos Norte, Aurora, Zambales, at Bataan, ay inaasahang makakaranas din ng malalakas na hangin at pagbuhos ng ulan.

## Mga Pangunahing Panganib at Babala
Mga kababayan, habang patuloy nating sinusubaybayan ang **Tropical Depression "Pepito,"** mahalaga pong malaman ang mga panganib na dapat nating ihanda:

**Una po, ang malalakas na hangin.** Ang mga hangin ay inaasahang may pagbuga na umaabot sa apatnapu't-lima kilometro bawat oras, at may pagbuga na umaabot naman sa limampu't-lima kilometro bawat oras, lalo na sa mga lugar malapit sa sentro ng bagyo.

**Pangalawa po, ang pagbuhos ng malalakas na ulan.** Ang PAGASA ay nagbabala ng inaasahang matinding pagbuhos ng ulan, lalo na sa Rehiyon ng Bicol at sa mga lalawigan ng Samar, Leyte, at ang mga katabing bahagi. Ang matinding ulan ay nagdudulot ng panganib ng pagbaha at pagguho ng lupa.

**Pangatlo po, ang karagatan.** Para sa mga nasa baybaying lugar, ang mga karagatan ay inaasahang magiging katamtaman hanggang malakas, na may taas na inaabot mula dalawang kuwarter hanggang apatnapu't limang sentimetro. Ito ay isang babala para sa mga naglalakbay sa dagat at mga komersyal na barko.

## Payo sa Kaligtasan at Paghahanda
Hinihikayat po ng PAGASA ang lahat ng mamamayan na maging handa.

Uulitin po natin ang kritikal na babala: **Ang paghahanda laban sa matinding pagbuhos ng ulan at malakas na hangin mula sa Tropical Depression "Pepito" ay kritikal.**

Hinihikayat po namin ang lahat na:
1.  **Maghanda ng Emergency Kits:** Siguraduhing may pagkain, malinis na tubig, first aid kit, at flashlight sa inyong tahanan.
2.  **Iwasan ang Paglalakbay:** Iwasang magbiyahe, lalo na sa mga kalsada o lugar na malapit sa baybayin, hangga’t hindi pa nagpapatuloy ang pag-uulat ng mas malakas na pagbabago.
3.  **Makinig sa Opisyal na Anunsyo:** Patuloy na makinig sa mga ulat mula sa PAGASA at sa inyong Local Government Unit.

Tandaan po natin na ang kaligtasan natin ang pinakamahalaga. Ang pagiging kalmado at maagap na pagkilos ang susi upang maiwasan ang anumang panganib na maidudulot ng **Tropical Depression "Pepito."**

## Pangwakas
Sa ngayon po, walang Tropical Cyclone Wind Signal na ipinapahiwatig. Gayunpaman, ipinapaalala po namin na nananatili tayong alerto.

Ang susunod po nating pag-uulat, isang bagong PAGASA Severe Weather Bulletin, ay itinakda pong magaganap sa alas-onse ng umaga. Manatiling nakatutok sa amin at sa mga opisyal na channel ng pamahalaan.

Maraming salamat po at mag-ingat po kayong lahat.

## Cebuano Version

**Language:** Cebuano  │  **Words:** 705  │  **Est. reading time:** 4.7 min

---

# TROPICAL DEPRESSION PEPITO: RADYO WEATHER ALERT BULLETIN

*(Sound effect: Short, gentle jingle music, tapos hinay-hinay nga mo-fade out)*

**Broadcaster:** Kumusta na mo diha, mga kauban! Malipayon kaayo ko nga makatambayayay ninyo pag-gabii niining atong programa. Karon, sama sa kanunay, ato gyud pang hisgutan ang pinaka-importanteng hisgutanan sa atong weather—ang mga update gikan ni PAGASA.

Mga kaigsoonan, kining pag-update nga atong madawat karon nag-tumong sa Tropical Depression nga giila og pangalan nga **PEPITO**. Busa, mga higala, pagpaminaw gyud mo og maayo kay importante kaayo ni nga mga detalye para sa inyong kaluwasan ug kasegurohan.

***

### Unsay Nahitabo Karon? (What's Happening Now)

Mga kauban, mga kaigsoonan, pag-andam kita nga atong i-follow og pag-ayo niining **PEPITO**. Sa pagkakaron, ang **PEPITO** usa ka Tropical Depression nga nag-form kogon sa pagkasadlunganan sa Eastern Catanduanes.

Ang maayong balita, mga kaigsoonan, mao nga kasamtangan, **walay Tropical Cyclone Wind Signal** ang gipahibalo sa publiko. Pero ato kining dili pasagdan nga maminaw lang. Kay bisan walay signal, nagpasabot gihapon ni og grabe nga mga kondisyon sa panahon.

Base sa forecast, ang **PEPITO** mo-migrate pa-west-northwest, unya mo-liko pa-westward padulong sa Northern Luzon. Ang mga eksperto sa PAGASA nag-ingon nga naa pa'y chance ang **PEPITO** nga molig-on pa—pwede pa siyang mag-upgrade ngadto sa usa ka Tropical Storm, o dili masayop, usa ka bagyo pa. Mao na ingon ka ka-importante, mga kauban.

***

### Asa Padulong ang Bagyo? (Where is it Heading)

Karon ha, atong atiman-an ang agianan niining **PEPITO**. Sa mga sunod nga oras, makita nato nga naglihok kini sa pag-ayo.

Mao kini ang forecast track, mga kauban:

*   Ug sa sunod kaadlawon, magpadulong kini sa mga bahin sa Aurora ug sa mga tubig east sa Cagayan.
*   Ug paglabay sa mga adlaw, ang **PEPITO** padulong pa-westward, padulong sa mga lugar sa Pangasinan.
*   Ang forecast, nagtandi sa pag-abot niini sa mga bahin sa Northern Luzon. Busa, mga kauban, kinahanglan gyud kitang mag-amping kon moabot na kini sa mga bahin sa Northern Luzon.

Ang kasagaran nga pag-forecast nagtudlo nga ang **PEPITO** makakaabot sa baybayon sa Eastern Coast sa Northern Luzon sa pagkahapon, sa adlaw nga ika-eh-kasi nga October. Busa, mag-andam gyud mo diha sa mga estorya sa Northern Luzon!

***

### Kinsa ang Apektado ug Unsa nga Danger? (Who is Affected & What is the Danger?)

Karon, ato pang hisgutan ang mga lugar nga pagadiyoha sa **PEPITO**. Dili lang usa ka lugar ang makaapekto. Daghang rehiyon ang kinahanglan gyud natong bantayan.

Mga kaigsoonan, ang pinaka-ma-apektado ug kinahanglan mag-andam gyud kay mao ni ang mga lugar sa **Catanduanes**, **Northern Samar**, **Samar**, ug tanan nga mga coastal area sa **Northern Luzon**.

Unsa man ang i-expect? Dili lang ni hangin, mga kauban.

Una, ang **uwan**. Ang pag-forecast nagtudlo nga adunay posibilidad nga **grabe kaayo nga pag-ulan** sa mga rehiyon sa Visayas, ilabi na sa gabii. Mao na ang atong pag-amping sa pagbaha.

Ikaduha, ang **hangin**. Nagpaabang na sa atong mga balita nga adunay kusog nga hangin, ang mga gusty nga mga hangin nga moabot sa mga upil ka-lima ka-kating-ka-metros matag oras. Busa, mga kauban, hugasi ang mga gamit nga pwede maghangin, i-secure ang mga kalit-kalit nga mga gamit sa balay.

Ug sa pinaka-importante, ang dagat. Para sa atong mga kaigsuon nga mangadto sa mga lugar nga coastal—daghang pag-amping! Mag-amping mo sa mga dagat, kay ang pag-forecast nag-ingon nga ang alon mahimong muabot og tunga hangtud sa upat ug katunga ka-kating-ka-metros. Dili ni dula-dula, mga kauban; grabe kaayo ni nga kusog sa dagat.

***

### Unsa ang Kinahanglan Buhaton? (What You Need to Do)

Busa, mga higala, kung makadungog mo niining mga pahibalo, ang atong mensahe mao kini: **Magpagawas og pag-andam!**

1.  **Pag-andam sa mga Pangutpa:** Aron sa atong mga kaigsuon nga anaa sa mga coastal areas, iandam ang mga rescue kits, tubig, pagkaon nga dali mokaon, ug mga first aid kit.
2.  **Dili mo'g dula-dula sa dagat:** Kung dili kinahanglan gyud nga mo-andar sa dagat o sa mga sapa, ayaw. I-secure ang inyong mga sakay ug i-monitor ang mga opisyal nga pahayag.
3.  **Pagpabilin nga update:** Ayaw pagpaminaw lang sa usa ka estasyon. Paminaw sa daghang mga opisyal nga istasyon.

Ang kasayon ug kaalam maoy atong pinakadakong kaalyado.

Tungod kay ang pag-monitor niining bagyo o pag-abot niining bagyo, magpabilin nga alerto. Mag-amping gyud mo sa inyong kaugalingon ug sa inyong mga pamilya.

Mabuhi kita tanan! Salamat kaayo sa pagpaminaw.

---
# PAGASA_22-TC02_Basyang_TCA#01
---

## English Version

**Language:** English  │  **Words:** 601  │  **Est. reading time:** 4.0 min

---

# Tropical Storm Malakas Advisory Update

*(Sound of gentle, continuous background music fades slightly)*

Good morning. This is your official weather bulletin update brought to you by the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA. We urge all our listeners to remain calm, stay informed, and remain vigilant as we monitor the movement of Tropical Storm **Malakas**.

***

## Current Situation

At this time, PAGASA advises that the center of **Tropical Storm Malakas** was last estimated at two point zero zero kilometers East of Mindanao. It is maintaining a respectable strength, with maximum sustained winds of seventy-five kilometers per hour, accompanied by higher gusts reaching up to ninety kilometers per hour. The storm is currently moving in a west-northwest direction, proceeding at a speed of fifteen kilometers per hour.

While **Tropical Storm Malakas** remains outside the Philippine Area of Responsibility, its movement and increasing intensity remain our primary concern for everyone in the maritime sector and those in surrounding regions.

***

## Forecast Track

Moving on to the forecast track, the data indicates a steady and concerning progression for the storm.

Over the next twenty-four hours, the forecast projects **Tropical Storm Malakas** to continue tracking generally in a west-northwest direction. By the following day, the estimated intensity increases significantly, and the storm begins to show characteristics of strengthening.

Looking further ahead—over the next seventy-two hours—the forecast models project **Tropical Storm Malakas** to potentially reach the classification of a Typhoon. The sustained winds are forecast to climb substantially, reaching an estimated one hundred fifty-five kilometers per hour by the next day after tomorrow. This strengthening trend is key information for all preparedness efforts. The forecast track suggests that the storm will gradually move eastward, passing far to the east of the Visayas, and then further east of Central Luzon.

***

## Affected Areas

Regarding affected areas, please remember that **Tropical Storm Malakas** is currently tracking significantly outside of our populated landmasses. Therefore, the immediate threat of tropical wind impact is low for most areas currently.

However, we advise all coastal communities, especially those in the waters between Mindanao and the Visayas, to remain highly alert. Sailors, fishing vessels, and all maritime operators must follow the advisories of the Philippine Coast Guard immediately. The ocean conditions surrounding the projected path of **Tropical Storm Malakas** are expected to be rough, with heavy swells and strong currents. Residents in the provinces of Northern Luzon and the areas adjacent to the potential track are advised to continue monitoring PAGASA advisories closely.

***

## Public Safety Advisory

Residents are urged to take proactive measures in anticipation of potential impacts. Since **Tropical Storm Malakas** shows clear signs of strengthening into a typhoon category, we must prepare for the highest possible weather conditions.

If PAGASA escalates the warning to a **Signal Number Three**, which indicates the greatest level of hazard, we repeat: prepare yourselves immediately. Always secure loose objects, trim trees, and ensure that all roofing materials are in good condition. Have your emergency kits ready, including flashlights, batteries, and sufficient potable water.

For those in low-lying areas, be prepared for potential storm surges and flooding, even if the storm center passes offshore. The most critical takeaway remains: stay tuned to local radio stations for continuous updates regarding **Tropical Storm Malakas**.

***

## Closing

We have provided the current situation and the longer-range forecast for **Tropical Storm Malakas**. Please remember to check the official PAGASA website for the most detailed charts and supplementary information.

We will provide our next detailed weather bulletin update at eleven o'clock in the morning. Until then, please take care, stay safe, and heed all local government advisories.

*(Sound of background music swells slightly)*

## Tagalog Version

**Language:** Tagalog  │  **Words:** 707  │  **Est. reading time:** 4.7 min

---

# TROPICAL STORM MALAKAS Advisory: Paghahanda at Pagbabantay

**(Simulan ang pagbasa sa mabagal, kalmado, at may awtoridad na tono.)**

Maligayang pagbati sa lahat ng nakikinig. Muli po kayong nakakasama sa isang espesyal na pagbababala sa panahon. Ito ay isang opisyal na pag-uulat mula sa **PAGASA**—ang Philippine Atmospheric, Geophysical, and Astronomical Services Administration.

Ang ating layunin po ngayon ay bigyan kayo ng pinakahuling kaalaman patungkol sa paggalaw at kalakasan ng Tropical Storm **MALAKAS**, upang lahat po tayo ay maging handa at mapagmatyag sa anumang pagbabago ng ating atmospera.

---

## Kasalukuyang Sitwasyon ng Tropical Storm Malakas

Batay sa pinakahuling datos na natanggap namin, ang **Tropical Storm MALAKAS** ay kasalukuyang matatagpuan sa karagatang silangan, sa labas po ng Philippine Area of Responsibility.

Sa oras na ito, tinatayang ang sentro ng bagyo ay nasa dalawang punto koma zero kilometro silangan ng Mindanao. Ang lakas nito ay nananatiling makapangyarihan. Ang tinatayang maximum sustained winds ay nasa pitumpu't limang kilometro bawat oras, na may kasamang hangin na posibleng umabot sa siyamnapu't kilometro bawat oras. Ang bilis ng paggalaw nito ay nasa labinlimang kilometro bawat oras, patungo sa direksyong Kanluran-Northwest.

Tandaan po natin, bagama’t nasa labas po ang bagyo ng aming ating mga pangunahing isla, hindi po natin dapat maliitin ang posibleng epekto nito sa ating mga panahon sa darating na mga araw.

## Forecast Track at Pagkakalala ng Bagyo

Patungo naman po tayo sa mas detalyadong forecast. Ang pagsubaybay sa **Tropical Storm MALAKAS** ay kritikal dahil patuloy itong nagpapakita ng potensyal para sa pagpapalakas habang patuloy itong gumagalaw pa-kanluran at pa-northwest.

Ayon sa mga pagtataya, inaasahang ang bagyo ay magpapatuloy sa paglalakbay sa mga sumusunod na bahagi ng dagat, at inaasahang mananatili itong sa labas ng ating makapangyarihang teritoryo.

Sa paglipas ng panahon, habang nagpapatuloy ang paggalaw nito, inaasahan na mananatili itong malakas at patuloy na makakakuha ng enerhiya mula sa mainit na tubig-dagat. Base sa pagtataya, mayroon tayong makikitang patuloy na paglala ng kalakasan ng bagyo. Sa loob ng animnapung oras, inaasahang ang lakas nito ay maaaring tumaas hanggang sa isang Typhoons na may tinatayang hangin na umaabot sa isang daan at limampu't limang kilometro bawat oras.

Ang pag-uulat na ito ay nagpapaalala po sa atin na ang malalakas na bagyo ay nagdadala ng mga posibleng pagbabago sa klima, kabilang na ang mas malalakas na hangin, malalakas na pagpatak ng ulan, at pagtaas ng tubig-dagat sa mga coastal areas.

## Mga Apektadong Lugar at Pangunahing Babala

Sa kasalukuyan, dahil ang sentro ng **Tropical Storm MALAKAS** ay nasa malayo at sa labas ng ating mga isla, ang agarang banta ng direktang pagtama sa anumang lalawigan ay hindi inaasahang mangyayari.

Gayunpaman, hinihiling po namin sa lahat ng residente na manatiling alerto. Ang mga bahagi ng ating bansa na matatagpuan sa mga coastal areas, at ang mga rehiyon na malapit sa mga inaasahang landas ng bagyo, ay dapat maghanda.

Dito po na magsisimula ang mahalagang paalala: **Ang paghahanda ay hindi nangangahulugan ng pag-aalala, kundi ng pagiging responsable.**

## Payo sa Kaligtasan at Paghahanda

Hinihikayat po ng PAGASA ang lahat ng residente na gawin ang mga sumusunod na hakbang bilang pag-iingat:

Una, **Manatiling Nakatutok.** Laging makinig sa opisyal na mga paalala mula sa inyong Local Government Unit o LGU. Huwag po tayong magpapadala sa mga hindi mapagkakatiwalaang impormasyon na po-internet.

Pangalawa, **Ihanda ang Inyong Emergency Kit.** Siguraduhin na mayroon kayong sapat na suplay ng inuming tubig, pagkain na hindi kailangan ng luto, first aid kit, at mga baterya.

Pangatlo, **Alamin ang mga Evacuation Center.** Alamin po ninyo ang pinakamalapit at pinakakaaasahang evacuation center sa inyong komunidad. Huwag pong mag-atubiling umalis o lumikas kung utusan po kayo ng mga awtoridad.

Ang pagiging handa po natin ang pinakamabisang depensa. Paulit-ulitin po nating tandaan: Kami po ay nananatiling nakatutok sa kalagayan ng **Tropical Storm MALAKAS**, at patuloy po tayong mag-iingat sa anumang pagbabago ng panahon.

## Pangwakas

Mga kababayan, ang pagbabantay sa **Tropical Storm MALAKAS** ay isang patuloy na proseso. Ang inyong pag-iingat at pagiging responsable ay makakatulong upang ang lahat ay manatiling ligtas.

Hinihiling po namin na patuloy kayong maging alerto, at patuloy po kayong sumusunod sa mga tagubilin ng inyong mga lokal na opisyal.

Muli po kaming mag-uulat sa inyo sa loob ng mga dalawang oras, o sa eksaktong alas-singko ng hapon. Manatiling kalmado, at mag-ingat po kayo. Maraming salamat sa inyong pakikinig.

## Cebuano Version

**Language:** Cebuano  │  **Words:** 694  │  **Est. reading time:** 4.6 min

---

# Tropical Storm Malakas Update: Pag-amping Gihapon sa mga Kauban

**(Sound of gentle, background radio jingle fades out)**

**BROADCASTER:** Maayong hapon, mga higala! Kumusta mo diha sa inyong mga balay?

Karon, mga kauban, mag-adto na ta sa atong weather update. Balo mo, ang pag-monitor sa atong mga palibot, labi na sa atong mga panglawas, mao gyud kana’y importante. Mao man na ang trabaho sa mga eksperto sa PAGASA, ang Philippine Atmospheric, Geophysical and Astronomical Services Administration. So, karon, atong hisgutan ang status sa bagyo nga gitawag og **Tropical Storm Malakas**.

Pero ayaw kabalaka una. Ang pag-monitor sa mga bagyo dili lang magpabug-at sa atong gibati; kini usa ka pagkaandam, usa ka pagkahibalo, mao na gyud.

***

## Unsay Nahitabo Karon (Current Situation)

Mga kauban, sumala sa latest advisory gikan sa PAGASA, ang **Tropical Storm Malakas** nagpabilin pa gihapon og kusog. Ang iyang sentro, or ang gitawag nato nga center, nakit-an sa pagka-duha ka kilometro sa Sidlakan sa Mindanao.

Karonha, mga higala, ang iyang maximum sustained winds gibana-lo nga napulo’g-walo ka kilometro kada oras. Karon man, ang iyang paglihok kay West-Northwest, sa kalendaryo sa mga kilometro kada oras.

Pero kining importante kaayo nga punto nga ato pong hisgutan, mga higala: sa pagtan-aw sa mga mapa ug sa pag-analyze sa direksyon, ang **Tropical Storm Malakas** karon, *wala pa gyud kaayo sa atong Philippine Area of Responsibility*—nag-giladlan pa siya og layo sa atong mga isla.

***

## Asa Padulong ang Malakas (The Forecast Path)

Karon ha, atong tan-awon ang *forecast*—kung asa na ni padulong.

Dili ni maoy pagdayeg, pero importante kaayo nga ipaabot nako sa inyo kining klaro nga imahe: Ang **Tropical Storm Malakas** padulong siya og layo.

Sa unang mga adlaw, samtang nag-agi pa siya og kasikbit sa atong mga atbang, mao nga kita mag-amping gihapon. Pero sa mga sunod nga mga adlaw, dako kaayo ang gilay-an niya sa atong mga isla.

Kung atong tan-awon ang forecast line, makita ninyo nga padulong kini paingon sa sidlakan, sa mas layo pa nga mga bahin sa dagat, nga kanunay gyud nga duol sa mga Visayas ug sa mga kasikbit sa pamaypay sa Luzon.

Ang pag-forecast sa bagyo, nga kasagaran nga pagpabato og mga mga direksyon, dili gyud mahimong 100 porsyento ang kasigurohan, pero ang patakaran sa mga eksperto kay nag-ingon nga padayon siyang molayo sa atong mga lugar.

***

## Kinsa ang Apektado (Who is Affected)

Ang atong mga kasalukuyang apektado, mga kauban, kay mao ra ang mga lugar nga atong kalikayan. Apan ang pag-adto sa bagyo dili makasalig.

Ang mga coastal areas, ang mga lugar nga duol sa baybayon, mao gyud kanunay ang mga nanginahanglan og pag-amping, bisag wala ka’y direkta nga pag-atubang sa kusog nga hangin.

Kini nagpasabot nga kinahanglan kita mag-andam sa mga epekto sa pag-ilis sa panahon: ang mga baha, ang mga pagtaas sa tubig sa dagat, ug ang mga hangin nga kusog, bisan wala man ka maoy sentro sa bagyo.

***

## Unsa ang Kinahanglan Buhaton (What to Do?)

Karon, ang akong mga tambag sa inyo, mga higala, dili lang kalipay ang pagdayeg sa pag-abot sa pag-ayo sa panahon, kundi ang *pagkaandam*.

Una, mga kauban, importante nga magpabilin kamo nga konektado sa mga opisyal nga mga source. Ayaw pagsalig sa mga *rumor* nga makit-an ninyo sa social media. Magpabiling updated sa mga balita sa lokal nga radyo o TV, o sa mga official PAGASA advisory.

Ikaduha, mag-andam og inyong **Go Bag** o disaster kit. Siguraduhon nga naa mo'y spare batteries, flashlight, portable charger, first-aid kit, ug mga pagkaon nga dili mahugot sa tubig.

Ug pinaka-importante, ang pagduol sa atong mga uban. Atong gi-check ang mga magulang, ang atong mga mga bata, ug ang atong mga kapitbahay. Ang pag-amping sa usag-usa kay usa ka pagpakita sa gugma.

***

## Pangwakas (Wrap Up)

So, mga kauban, sa pagtapos niining atong weather talk, atong i-summarize: ang **Tropical Storm Malakas** nagpadayon, apan ang iyang direksyon nagpasabot nga padulong siya sa paglayo gikan sa atong mga lugar. Pero, dili gyud ta makapahulay! Magpabilin kitang alerto.

Kita mag-update pag-usab sa atong mga weather condition ug sa mga advisories sa PAGASA. Magkita ta’g usab sa atong himoon nga pag-ambit sa kaayohan.

Hangtod karon, pag-amping kamo sa inyong mga pamilya, ug kanunay mag-ingon: *Alerto, pero kalma!*

**(Sound of radio jingle fades in)**

## 5. Length Check

Verify word counts across all three languages are close to the 750-word target. If any are significantly short or long, we can adjust the prompts.

In [8]:
TARGET_WORDS = 750
WORDS_PER_MIN = 150

print("\nLENGTH SUMMARY")
print("=" * 60)
print(f"{'Bulletin':<35} {'Lang':>4} {'Words':>6}  {'Minutes':>7}  {'vs Target':>10}")
print("-" * 60)

for result in results:
    diff = result['word_count'] - TARGET_WORDS
    diff_str = f"+{diff}" if diff >= 0 else str(diff)
    label = result['stem'].replace('PAGASA_', '')
    lang = result['language'].upper()
    print(f"{label:<35} {lang:>4} {result['word_count']:>6}  {result['reading_minutes']:>6.1f}m  {diff_str:>10}")

print("-" * 60)
print(f"Target: {TARGET_WORDS} words ≈ {TARGET_WORDS/WORDS_PER_MIN:.0f} minutes at {WORDS_PER_MIN} wpm")
print(f"\n✓ {len(results)} scripts saved to: {output_dir.absolute()}")


LENGTH SUMMARY
Bulletin                            Lang  Words  Minutes   vs Target
------------------------------------------------------------
20-19W_Pepito_SWB#01                  EN    608     4.1m        -142
20-19W_Pepito_SWB#01                  TL    697     4.6m         -53
20-19W_Pepito_SWB#01                 CEB    705     4.7m         -45
22-TC02_Basyang_TCA#01                EN    601     4.0m        -149
22-TC02_Basyang_TCA#01                TL    707     4.7m         -43
22-TC02_Basyang_TCA#01               CEB    694     4.6m         -56
------------------------------------------------------------
Target: 750 words ≈ 5 minutes at 150 wpm

✓ 6 scripts saved to: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins
